# 第 8 章 多変数の微分と効用最大化: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/math-for-economics-programming/08-partial-derivatives-and-utility-maximization.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
price_x = 2     # 財 x の価格 p_x（正）
price_y = 5     # 財 y の価格 p_y（正）
income = 100    # 所得 m（0 以上）
share = 0.5     # 指数 a（U = x^a·y^(1−a)。0.05〜0.95。1 に近いと図が描けない）
weight = 4      # 準線形効用 U = b·√x + y の重み b（正）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## コブ・ダグラス型の効用最大化

In [ ]:
from lib.chapter08.utility import (
    best_bundle_on_budget_line,
    cobb_douglas_demand,
    cobb_douglas_lagrange_multiplier,
    cobb_douglas_mrs,
    cobb_douglas_utility,
    quasilinear_demand,
    quasilinear_utility,
)

prices = [price_x, price_y]
x, y = cobb_douglas_demand(prices, income, share)
print(f"最適点（解析解）: x = {x:.3f}、y = {y:.3f}、効用 U = "
      f"{cobb_douglas_utility(x, y, share):.3f}")
if income > 0:
    print(f"限界代替率 {cobb_douglas_mrs(x, y, share):.3f}、"
          f"価格比 {price_x / price_y:.3f}")
    print(f"ラグランジュ乗数（1 円あたりの限界効用）"
          f"{cobb_douglas_lagrange_multiplier(prices, income, share):.4f}")
else:
    print("所得が 0 では、最適点は (0, 0) で、限界代替率は決まりません")
grid_x, grid_y = best_bundle_on_budget_line(
    lambda u, v: cobb_douglas_utility(u, v, share), prices, income, 2001
)
print(f"格子探索（2001 点）: x = {grid_x:.3f}、y = {grid_y:.3f}")

## 準線形効用 U = b·√x + y の最適点

In [ ]:
qx, qy = quasilinear_demand(prices, income, weight)
kind = "端点解（y = 0）" if qy == 0 else "内側の解"
print(f"x = {qx:.3f}、y = {qy:.3f}（{kind}）、効用 U = "
      f"{quasilinear_utility(qx, qy, weight):.3f}")

## 図

In [ ]:
from lib.chapter08.plot import plot_income_expansion_paths, plot_utility_maximization

incomes = [income * k / 5 for k in range(1, 6)]
if income > 0:  # 所得が 0 では、接点の図は描けない（限界効用が決まらない）
    display(plot_utility_maximization(prices, income, share))
display(plot_income_expansion_paths(prices, incomes, share, weight))

## 試してみよう

- `share` を 0.3 にすると、最適点は x = 15.000、y = 14.000 に変わります（所得の 3 割を
  x に使う）。限界代替率は 0.400 のままで、価格比に等しいことを確かめてください
- `price_x` を 4 にすると、x は 12.500 に半減し、y は 10.000 のままです（a = 0.5 では、
  x への支出が所得の半分のまま）。ラグランジュ乗数は 0.1118 に下がります
- `income` を 200 にすると、コブ・ダグラス型の x は 25.000 から 50.000 に倍増しますが、
  準線形効用の x は 25.000 のままで、増えた所得はすべて y（10.000 → 30.000）に回ります
- `income` を 10 にすると、準線形効用の最適点は x = 5.000、y = 0.000 の**端点解**です
  （x = 25 を買えないので、所得をすべて x に使う）